# Zel'dovich analysis on enzo

In [13]:
import yt   # library to upload and analyze AMR
import numpy as np
import matplotlib.pyplot as plt
import os
import glob
from unyt import Mpc, km, s, K, g, cm 
import h5py

In [14]:
# Physical paramters
h = 0.7
H0 = (h * 100.0 * km / s / Mpc).to('1/s')   # convesion in 1/s
a_init = 0.02                               # initial condition for the scale factor

In [15]:
def get_a(t_sim):
    t_init = (2.0/3.0) * (1.0/H0) * (a_init**1.5)     # age of the universe at a=0.02 (z=49)
    t_abs = t_sim + t_init                            # time from the BB in the simulation
    return (1.5 * H0 * t_abs)**(2.0/3.0)              # scale factor for each time of the simulation


# Import the plotfile

path = '/data/mfulghieri/enzo-dev/run/Cosmology/ZeldovichPancake/DD0021'   # final plotfile

# create the dataset
ds = yt.load(path)

YTUnidentifiedDataType: Could not determine input format from '/data/mfulghieri/enzo-dev/run/Cosmology/ZeldovichPancake/DD0021'
The following types could not be thorougly checked against your data because their requirements are missing. You may want to inspect this list and check your installation:
- CFRadialDataset (requires: xarray, pyart)
- CM1Dataset (requires: netCDF4)
- EnzoEDataset (requires: libconf)
- EventsFITSDataset (requires: astropy)
- ExodusIIDataset (requires: netCDF4)
- FITSDataset (requires: astropy)
- SkyDataFITSDataset (requires: astropy)
- SpectralCubeFITSDataset (requires: astropy)
- YTFITSDataset (requires: astropy)

Please make sure you are running a sufficiently recent version of yt.

In [17]:


# open the HDF5 file in read-only mode
path = '/data/mfulghieri/enzo-dev/run/Cosmology/ZeldovichPancake/DD0021'   # final plotfile

with h5py.File(path, 'r') as f:

  # file inspection
  print(f"The h5py file contains objects: {f.keys()}, with the datasets: {f['PartType0'].keys()}")
  print("Groups:", list(f.keys()))
  print("-" * 50)


  # ----------------------- Metadata -----------------------

  # inspect the header (metadata)
  header = f['Header'].attrs

  # print the relevant metadata
  print(f"\nRelevant Metadatas")
  print(f"\nUnitLength_in_cm: {header['UnitLength_in_cm']}")
  print(f"UnitMass_in_g: {header['UnitMass_in_g']}")
  print(f"UnitVelocity_in_cm_per_s: {header['UnitVelocity_in_cm_per_s']}")
  print(f"Hubble parameter: {header['HubbleParam']}")
  print(f"Redshift of the snapshot: {header['Redshift']}")
  print(f"Time of the snapshot: {header['Time']}")
  print(f"Omega0: {header['Omega0']}")

  print(f"Box size in code units: {header['BoxSize']}")     # BoxSize


  # conversion factors
  kpc_to_cm = header['UnitLength_in_cm']                 # 3.085678e+21 cm/kpc
  ten_solar_mass_to_g = header['UnitMass_in_g']          # 1.989e+33 g/Msun
  kms_to_cms = header['UnitVelocity_in_cm_per_s']        # 100000.0 (cm/s) / (km/s)


  # see all the metadata (commented to avoid mess)
  # for key, val in header.items():
  #  print(f"{key}: {val}")


  # store the relevant header attributes
  hubble_param      = header['HubbleParam']    # for the velocity calculation and for units conversion (simulations usually use coomoving coordinates)
  redshift          = header['Redshift']       # z = 2.464
  scale_factor      = header['Time']           # a = 0.2886 at this redshift
  Lbox              = header['BoxSize']        # 25000.0 kpc, box size of Illustris TNG (units of kpc / (h * a))
  omega_m           = header['Omega0']         # 0.3 (flat universe with only Lambda and matter)
  omega_l           = header['OmegaLambda']    # 0.7 (flat universe with only Lambda and matter)


  # conversion of Lbox in physical cgs units
  Lbox = Lbox * kpc_to_cm * scale_factor / hubble_param


  # define the scale factor a from the redshift of the snapshot (NB: it is exactly the time)
  print(f"\nCheck that the time in the header is the scale factor at that redshift: time - 1 / (1 + redshift) = {scale_factor - 1 / (1 + redshift)}")
  print("-" * 50)



  # ----------------------- Gas Data -----------------------
  # OBS: [:] to extraxt the data also out of this scope of with
  # import the data and convert to physical units in CGS


  # DATA IMPORTATION AND CONVERSION IN PHYS COORDS


  # 1. Positions: A 3D array (N, 3) in units of kpc * a / h of the coomving positions
  coordinates = f['PartType0/Coordinates'][:] * kpc_to_cm * scale_factor / hubble_param


  # 2. Masses: converted in solar mass units/h (10M_sun/h), where 10M_sun = 1.989e+43 g
  masses = f['PartType0/Masses'][:] * ten_solar_mass_to_g / hubble_param


  # 3. Metallicities: Total metal mass fraction (Z) and individual elements
  metallicity = f['PartType0/GFM_Metallicity'][:]


  # 4. Densities: local gas density (10^10 M_sun h^2/kpc^3)
  density = f['PartType0/Density'][:] * ten_solar_mass_to_g * hubble_param**2 / (kpc_to_cm**3 * scale_factor**3)


  # 5. Velocities in 100000.0 cm/s (1km/s) (peculiar velocities)
  velocity = f['PartType0/Velocities'][:] * kms_to_cms * np.sqrt(scale_factor)


  # 6. Internal energies (for the temperature). Units: energy/mass = vel^2 [10^5 cm/s]^2 = 10^10 cm^2/s^2
  internal_energy = f['PartType0/InternalEnergy'][:] * kms_to_cms**2   # converted in cm^2/s^2


  # 7. Electron abundance (to calculate the mean molecular weight) x_e = n_e/n_H
  # the mean molecular weight mu is the average mass per particle (atoms + ions + e-) in units of proton mass
  electron_abundance = f['PartType0/ElectronAbundance'][:]           # (16220782,) array


  # 8. Smoothing length converted from kpc to cm
  h_smoothing = f['PartType0/SubfindHsml'][:] * kpc_to_cm * scale_factor / hubble_param          # (16220782,) array with the smoothing length of each gas particle


  # 9. Neutral Hydrogen abundance: density of the Lyalpha absorber. Fraction of the HI density with respect to the total H density (rho_HI/rho_H)
  HI_abundance = f['PartType0/NeutralHydrogenAbundance'][:]



Npart = len(masses)
print(f"\nSuccessfully loaded {Npart} gas particles.")

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '/data/mfulghieri/enzo-dev/run/Cosmology/ZeldovichPancake/DD0021/data0000', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)